# Policy-Gradient Experiment Guide

This notebook provides a code-only workflow for comparing REINFORCE, REINFORCE with a baseline, Actor-Critic, and A2C on CartPole-v1. It is intentionally not executed before committing. Run the cells locally after adjusting the configuration and inspect the generated TensorBoard logs.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

import gymnasium as gym
import numpy as np

POLICY_GRADIENT_DIR = Path.cwd()
if POLICY_GRADIENT_DIR.name != 'policy-gradient':
    POLICY_GRADIENT_DIR = Path('/home/dangineer/Learnspaces/coursera/reinforcement-learning-specialization/rl-lab/deep-rl/policy-gradient')
DEEP_RL_DIR = POLICY_GRADIENT_DIR.parent
for directory in (DEEP_RL_DIR, POLICY_GRADIENT_DIR):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from actor_critic import ActorCritic
from a2c import A2CAgent
from reinforce import REINFORCE
from reinforce_with_baseline import REINFORCE as REINFORCEWithBaseline

## 1. Configure the experiment

Start with a small number of episodes while debugging. Increase the values for longer comparisons. The notebook defines the complete experiment but does not run it automatically.

In [ ]:
ENV_ID = 'CartPole-v1'
GLOBAL_SEED = 2026
NUM_SEEDS = 3
MAX_EPISODES = 1_000
MAX_EPISODE_STEPS = 500
A2C_NUM_ENVS = 16
A2C_MAX_STEPS = 200_000
A2C_N_STEPS = 10
LOG_DIR = POLICY_GRADIENT_DIR / 'runs' / 'policy_gradient_comparison'

seed_generator = np.random.default_rng(GLOBAL_SEED)
SEEDS = [int(seed) for seed in seed_generator.integers(0, 2**31 - 1, size=NUM_SEEDS)]

print(f'Environment: {ENV_ID}')
print(f'Global seed: {GLOBAL_SEED}')
print(f'Experiment seeds: {SEEDS}')
print(f'Episodes per agent: {MAX_EPISODES}')
print(f'Maximum episode steps: {MAX_EPISODE_STEPS}')
print(f'TensorBoard log directory: {LOG_DIR}')

## 2. Define the environment and experiment runners

Each run receives a fresh environment and seed. The TimeLimit wrapper keeps episode lengths consistent across agents.

In [ ]:
def make_env(seed):
    env = gym.wrappers.TimeLimit(
        gym.make(ENV_ID),
        max_episode_steps=MAX_EPISODE_STEPS,
    )
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env


def make_vector_env(seed, num_envs):
    def factory(index):
        def thunk():
            env = gym.wrappers.TimeLimit(
                gym.make(ENV_ID),
                max_episode_steps=MAX_EPISODE_STEPS,
            )
            env.reset(seed=seed + index)
            env.action_space.seed(seed + index)
            return env
        return thunk
    return gym.vector.SyncVectorEnv([factory(index) for index in range(num_envs)])

agent_specs = {
    'reinforce': REINFORCE,
    'reinforce_with_baseline': REINFORCEWithBaseline,
    'actor_critic': ActorCritic,
}


def run_episode_agent(agent_name, agent_type):
    completed_runs = []
    for seed in SEEDS:
        env = make_env(seed)
        agent = agent_type(
            state_dim=env.observation_space.shape[0],
            action_dim=env.action_space.n,
            seed=seed,
            tensorboard_log_dir=str(LOG_DIR),
            run_name=agent_name,
        )
        agent.learn(env, max_episodes=MAX_EPISODES)
        agent.close()
        env.close()
        completed_runs.append((agent_name, seed))
        print(f'{agent_name}, seed={seed}: training complete')
    return completed_runs

## 3. Run the comparison

Run this cell only when you are ready to train all configured agents. It writes TensorBoard event files under LOG_DIR.

In [ ]:
completed_runs = []
for agent_name, agent_type in agent_specs.items():
    completed_runs.extend(run_episode_agent(agent_name, agent_type))

for seed in SEEDS:
    envs = make_vector_env(seed, A2C_NUM_ENVS)
    agent = A2CAgent(
        state_dim=envs.single_observation_space.shape[0],
        num_actions=envs.single_action_space.n,
        n_steps=A2C_N_STEPS,
        seed=seed,
    )
    agent.learn(envs, max_steps=A2C_MAX_STEPS)
    agent.close()
    envs.close()
    completed_runs.append(('a2c', seed))
    print(f'a2c, seed={seed}: training complete')

print(f'Completed runs: {len(completed_runs)}')

## 4. Inspect logs with TensorBoard

After training, launch TensorBoard from this notebook or from a terminal. Compare episode returns, episode lengths, and training losses across agents and seeds.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/policy_gradient_comparison

## 5. Hyperparameter tuning

- Tune policy and value learning rates separately; the critic often uses a larger learning rate.
- Adjust gamma to control the importance of future rewards.
- Increase network capacity only when the observation space or task requires it.